In [3]:
import mlflow
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np


/home/vcrostin/anaconda3/envs/mlbegin/lib/python3.12/site-packages/pydantic/_internal/_config.py:373: UserWarning: Valid config keys have changed in V2:
* 'schema_extra' has been renamed to 'json_schema_extra'
  warnings.warn(message, UserWarning)


In [4]:
import os
os.environ["MLFLOW_ENABLE_ASYNC_LOGGING"] = "true"


diabetes = load_diabetes()
X, y = diabetes.data, diabetes.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [12]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', RandomForestRegressor(n_estimators=100, random_state=42))
])

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("diabetes-prediction")

with mlflow.start_run():
    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("random_state", 42)
    mlflow.log_metric("mse", mse)
    mlflow.log_metric("r2_score", r2)

    mlflow.sklearn.log_model(pipeline, "diabets-model", input_example=X_train, registered_model_name="diabetes-prediction")

print(f"MSE: {mse:.2f}")
print(f"R2 Score: {r2:.2f}")


Successfully registered model 'diabetes-prediction'.
2026/03/14 18:17:58 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: diabetes-prediction, version 1


MSE: 2959.18
R2 Score: 0.44


Created version '1' of model 'diabetes-prediction'.


In [15]:
print("Тестирование загрузки модели:")
model_uri = "models:/diabetes-prediction/1"
loaded_model = mlflow.sklearn.load_model(model_uri)

test_sample = X_test[0:1]
prediction = loaded_model.predict(test_sample)
print(f"Тестовое предсказание: {prediction[0]:.2f}")

Тестирование загрузки модели:


Тестовое предсказание: 146.21
